In [110]:
#  Practical Assignment: Data Preprocessing Pipeline - Lesson 3

# 1. Import the necessary libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

print("Successfully imported libraries")


Successfully imported libraries


In [111]:
# 1 - Load & Inspect Dataset - Car Prices
# === LOAD DATASET FROM CSV FILE
CSV_FILE_PATH = "raw_car_dataset.csv"
df = pd.read_csv(CSV_FILE_PATH)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [112]:
# === INITIAL SNAPSHOT
print("\n===  INITIAL HEAD:")
print(df.head())


===  INITIAL HEAD:
    Price  Odometer_km  Doors  Accidents Location  Year
0  $1,500     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  $5,331     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022


In [113]:
print("\n===  INITIAL INFO:")
print(df.info())


===  INITIAL INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    object 
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accidents    145 non-null    int64  
 4   Location     140 non-null    object 
 5   Year         145 non-null    int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 6.9+ KB
None


In [114]:
print("\n===  INITIAL SHAPE:")
print(df.shape)


===  INITIAL SHAPE:
(145, 6)


In [115]:
print("\n===  INITIAL MISSING VALUES:")
print(df.isnull().sum())


===  INITIAL MISSING VALUES:
Price          0
Odometer_km    7
Doors          7
Accidents      0
Location       5
Year           0
dtype: int64


In [116]:
# === COUNT UNIQUE VALUES
print("\n===  COUNT UNIQUE VALUES:")
print(df.nunique())



===  COUNT UNIQUE VALUES:
Price           56
Odometer_km    132
Doors            4
Accidents        4
Location         5
Year            26
dtype: int64


In [117]:
# === LOCATION COUNTS
print("\n===  LOCATION COUNTS:")
print(df["Location"].value_counts(dropna=False))



===  LOCATION COUNTS:
Location
City      59
Suburb    45
Rural     21
Subrb      8
??         7
NaN        5
Name: count, dtype: int64


In [118]:
# 2 - Clean Target Formatting (Price)  - remove currency/commas; ensure numeric; report dtype and skewness.
df["Price"] = df["Price"].str.replace("[$,]", "", regex=True).astype(float)
print("\n===  PRICE DATATYPE AND SKEWNESS:")
print("Price Datatype:", df["Price"].dtype)
print("Price Skewness:" ,df['Price'].skew())


===  PRICE DATATYPE AND SKEWNESS:
Price Datatype: float64
Price Skewness: 7.871113660850301


In [119]:
# 3 - Fix Category Errors before Imputation — normalize Location text, map typos (e.g., Subrb→Suburb), convert unknowns (e.g., “??”) to missing; recount including missing.
df['Location'] =df ["Location"].replace({"Subrb": "Suburb", "??": pd.NA})
print("\n===  LOCATION COUNTS AFTER FIXES:")
print(df["Location"].value_counts(dropna=False))


===  LOCATION COUNTS AFTER FIXES:
Location
City      59
Suburb    53
Rural     21
<NA>       7
NaN        5
Name: count, dtype: int64


In [120]:
# 4 -Impute Missing Values (justify choices) — Odometer_km→median; Doors/Accidents→mode; Location→mode; confirm post-imputation missing counts.
df["Odometer_km"] = df["Odometer_km"].fillna(df["Odometer_km"].median())
df["Doors"] = df["Doors"].fillna(df["Doors"].mode()[0])
df["Accidents"] = df["Accidents"].fillna(df["Accidents"].mode()[0])
df["Location"] = df["Location"].fillna(df["Location"].mode()[0])

print("\n===  MISSING VALUES AFTER IMPUTATION:")
print(df.isnull().sum())


===  MISSING VALUES AFTER IMPUTATION:
Price          0
Odometer_km    0
Doors          0
Accidents      0
Location       0
Year           0
dtype: int64


In [121]:
# 5 - Remove Duplicates — report shape before/after and rows removed.
Before = df.shape
df= df.drop_duplicates()
After = df.shape
print("\n===  SHAPE BEFORE AND AFTER DUPLICATE REMOVAL:")
print("Before:", Before)
print("After:", After)


===  SHAPE BEFORE AND AFTER DUPLICATE REMOVAL:
Before: (145, 6)
After: (140, 6)


In [122]:
# 6 - Outliers (IQR capping) — compute bounds and clip for Price and Odometer_km; show a short summary after capping.
def iqr_function(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper
price_lower, price_upper = iqr_function(df["Price"])
odometer_lower, odometer_upper = iqr_function(df["Odometer_km"])
df["Price"] = df["Price"].clip(price_lower, price_upper)
df["Odometer_km"] = df["Odometer_km"].clip(odometer_lower, odometer_upper)

print("\n===  PRICE AND ODOMETER_KM SUMMARY AFTER CAPPING:")
print(df[["Price", "Odometer_km"]].describe())


===  PRICE AND ODOMETER_KM SUMMARY AFTER CAPPING:
             Price    Odometer_km
count   140.000000     140.000000
mean   3175.456250  130945.403571
std    2601.848629   53815.006935
min    1500.000000    5000.000000
25%    1500.000000   97844.000000
50%    1500.000000  128548.000000
75%    4489.750000  167501.500000
max    8974.375000  271987.750000


In [123]:
# 7 - One-Hot Encode Categorical(s) — encode Location as 0/1 columns; list the new columns created.
df = pd.get_dummies(df, columns=["Location"], drop_first=False, dtype="int")
print("\n===  NEW COLUMNS AFTER ONE-HOT ENCODING LOCATION:")
print([col for col in df.columns if col.startswith("Location_")])



===  NEW COLUMNS AFTER ONE-HOT ENCODING LOCATION:
['Location_City', 'Location_Rural', 'Location_Suburb']


In [124]:
# 8 - Feature Engineering (no leakage) — add at least three sensible features (e.g., CarAge, Km_per_year with safe handling, Is_Urban). Add LogPrice = log(Price+1) as an alternative target (not a feature).
df["CarAge"] = 2024 - df["Year"]
df["Km_per_year"] = df["Odometer_km"] / df["CarAge"].replace(0, 1)  # Avoid division by zero
df["Is_Urban"] = np.where(df[[col for col in df.columns if col.startswith("Location_")]].sum(axis=1) > 0, 1, 0)
df["LogPrice"] = np.log1p(df["Price"])
print("\n===  NEW FEATURES SAMPLE:")
print(df[["CarAge", "Km_per_year", "Is_Urban", "LogPrice"]].head())


===  NEW FEATURES SAMPLE:
   CarAge   Km_per_year  Is_Urban  LogPrice
0      26   5301.153846         1  7.313887
1       8  16068.500000         1  8.336151
2      10  10730.200000         1  8.581482
3      25   5673.520000         1  7.313887
4       2  64274.000000         1  7.313887


In [125]:
# 9 - Feature Scaling (X only) — standardize continuous features; do not scale Price or LogPrice; prefer leaving 0/1 dummies unscaled; show a small sample of scaled values.
scaler = StandardScaler()
df[["Odometer_km", "CarAge", "Km_per_year"]] = scaler.fit_transform(df[["Odometer_km", "CarAge", "Km_per_year"]])
print("\n===  SCALED FEATURES SAMPLE:")
print(df[["Odometer_km", "CarAge", "Km_per_year"]].head())


===  SCALED FEATURES SAMPLE:
   Odometer_km    CarAge  Km_per_year
0     0.128390  1.686714    -0.439793
1    -0.044709 -0.794617    -0.098297
2    -0.440923 -0.518913    -0.267606
3     0.203135  1.548862    -0.427983
4    -0.044709 -1.621727     1.430580


In [126]:
#  +10 -Final Checks & Save — final info, missing counts (all zero), a small describe table; save to car_l3_clean_ready.csv.
print("\n===  FINAL HEAD:")
print(df.head())

print("\n===  FINAL INFO:")
print(df.info())

print("\n===  FINAL MISSING VALUES:")
print(df.isnull().sum())

print("\n===  FINAL DESCRIBE:")
print(df.describe())

OUT_PATH = 'clean_car_dataset.csv'
df.to_csv(OUT_PATH, index=False)
print(f"\nCleaned dataset saved to {OUT_PATH}")


===  FINAL HEAD:
    Price  Odometer_km  Doors  Accidents  Year  Location_City  Location_Rural  \
0  1500.0     0.128390    4.0          1  1998              1               0   
1  4171.0    -0.044709    4.0          0  2016              0               1   
2  5331.0    -0.440923    4.0          0  2014              0               0   
3  1500.0     0.203135    4.0          1  1999              0               0   
4  1500.0    -0.044709    3.0          0  2022              1               0   

   Location_Suburb    CarAge  Km_per_year  Is_Urban  LogPrice  
0                0  1.686714    -0.439793         1  7.313887  
1                0 -0.794617    -0.098297         1  8.336151  
2                1 -0.518913    -0.267606         1  8.581482  
3                1  1.548862    -0.427983         1  7.313887  
4                0 -1.621727     1.430580         1  7.313887  

===  FINAL INFO:
<class 'pandas.core.frame.DataFrame'>
Index: 140 entries, 0 to 139
Data columns (total 12 col

             Price   Odometer_km       Doors   Accidents         Year  \
count   140.000000  1.400000e+02  140.000000  140.000000   140.000000   
mean   3175.456250  3.172066e-18    3.785714    0.721429  2010.235714   
std    2601.848629  1.003591e+00    0.846369    0.882017     7.280219   
min    1500.000000 -2.348743e+00    2.000000    0.000000  1998.000000   
25%    1500.000000 -6.173048e-01    3.000000    0.000000  2004.000000   
50%    1500.000000 -4.470894e-02    4.000000    0.000000  2010.000000   
75%    4489.750000  6.817310e-01    4.000000    1.000000  2016.250000   
max    8974.375000  2.630285e+00    5.000000    3.000000  2023.000000   

       Location_City  Location_Rural  Location_Suburb        CarAge  \
count     140.000000      140.000000       140.000000  1.400000e+02   
mean        0.485714        0.150000         0.364286 -9.516197e-18   
std         0.501590        0.358354         0.482957  1.003591e+00   
min         0.000000        0.000000         0.000000 -1.7